# 01. Potential Outcomes & DAGs

## Background

Causal inference asks a fundamentally different question than prediction: not "what does X predict?" but "what happens to Y if we intervene and *set* X?" The distinction matters enormously. A model trained to predict hospital readmissions from vitals will happily exploit spurious correlates — sicker patients get more tests, so test counts predict readmission. Intervening on test counts changes nothing about readmission rates. Confusing prediction with causation is how well-intentioned policies backfire.

Two complementary frameworks dominate modern causal inference: the **Potential Outcomes** framework (Rubin 1974, Neyman 1923) and the **Structural Causal Model / DAG** framework (Pearl 1995, 2009). They are mathematically equivalent but illuminate different aspects of the problem. Potential outcomes are concrete — you can almost write them down in a spreadsheet. DAGs are graphical — you can draw them, show them to domain experts, and reason about identification visually.

## What You'll Learn

- The potential outcomes model: Y(0), Y(1), and the fundamental problem of causal inference
- SUTVA: what it assumes and when it breaks
- ATE, ATT, ATC — which estimand you actually want
- DAG notation: nodes, directed edges, paths, forks, chains, colliders
- Backdoor criterion (narrative): when observed covariates are sufficient to identify ATE
- Building and visualizing DAGs with NetworkX

## Why This Matters

Every causal method in this series — propensity scoring, DiD, RD, IV, Double ML — is an answer to the same question: "which identification strategy licenses us to interpret this statistical estimate as a causal effect?" Potential outcomes give us the language to define what we're estimating. DAGs tell us whether we can estimate it from observational data — and which variables to condition on (and which never to touch).

The community standard (Hernán & Robins 2020, Imbens & Rubin 2015, Peters et al. 2017) is to make the identification assumptions explicit before touching the data. A DAG is a commitment device: draw it, defend it, then pick your estimator.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from itertools import product

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Potential Outcomes Framework

For each unit *i*, define:
- **Y_i(1)**: the outcome *if* treated
- **Y_i(0)**: the outcome *if* not treated

The **individual treatment effect** is τ_i = Y_i(1) − Y_i(0).

The **fundamental problem of causal inference**: we only ever observe *one* of these outcomes. The counterfactual is missing.

In [ ]:
# Simulate the "science table" — the unobserved full data
np.random.seed(0)
n = 12

# True potential outcomes (god's eye view, never observable)
y0 = np.random.normal(5, 1.5, n).round(2)   # untreated potential outcome
te = np.random.normal(2, 0.8, n).round(2)    # individual treatment effect
y1 = (y0 + te).round(2)                      # treated potential outcome

# Assignment mechanism (RCT: coin flip)
treated = np.random.binomial(1, 0.5, n).astype(bool)

# Observed outcome: the factual potential outcome
y_obs = np.where(treated, y1, y0)

science_table = pd.DataFrame({
    'Unit': range(1, n+1),
    'Y(0)': y0,
    'Y(1)': y1,
    'τ_i':  te,
    'Treated': treated.astype(int),
    'Y_obs':   y_obs,
    'Y_missing': np.where(treated, y0, y1),  # counterfactual — never seen
})

# Mask counterfactuals (what researchers actually see)
observed_df = science_table[['Unit','Treated','Y_obs']].copy()
observed_df['Y_missing'] = '?'

print("=== Science Table (God's Eye View) ===")
print(science_table.to_string(index=False))
print(f"\nTrue ATE  = E[Y(1)-Y(0)] = {te.mean():.3f}")
print(f"True ATT  = E[Y(1)-Y(0) | T=1] = {te[treated].mean():.3f}")
print(f"\nNaive DiM = E[Y|T=1] - E[Y|T=0] = {y_obs[treated].mean() - y_obs[~treated].mean():.3f}")
print("(DiM ≈ ATE here because assignment is random)")

In [ ]:
# Show selection bias with observational data
# Confounded assignment: sicker people more likely treated
np.random.seed(1)
health = np.random.normal(0, 1, n)         # unobserved health (-ve = sicker)
p_treat = 1 / (1 + np.exp(-(-health)))    # sicker → more likely treated
treated_conf = np.random.binomial(1, p_treat).astype(bool)

# True effect is +2 for everyone
y0_conf = 5 + health * 1.5 + np.random.normal(0, 0.5, n)
y1_conf = y0_conf + 2.0
y_obs_conf = np.where(treated_conf, y1_conf, y0_conf)

naive_estimate = y_obs_conf[treated_conf].mean() - y_obs_conf[~treated_conf].mean()
print(f"Confounded setting:")
print(f"  True ATE  = 2.000")
print(f"  Naïve DiM = {naive_estimate:.3f}   ← selection bias")
print(f"  Treated group is sicker → lower baseline → naive estimate biased downward")

## 2. SUTVA — Stable Unit Treatment Value Assumption

SUTVA has two components:
1. **No interference**: my treatment doesn't affect your outcome (violated in networks, vaccines)
2. **No hidden versions of treatment**: "treated" means the same thing for everyone

When SUTVA fails, Y_i(1) depends on others' treatment status and the whole framework breaks. Partial equilibrium effects (supply shocks, wage subsidies) are classic violations.

In [ ]:
# SUTVA violation illustration — network spillovers
# In a peer network, vaccinating some units protects their neighbors
# So Y_i(0) depends on treatment status of neighbors → potential outcomes are not unit-specific

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Clustered network
G = nx.karate_club_graph()
pos = nx.spring_layout(G, seed=7)
treated_nodes = list(range(0, 10))  # first cluster treated

node_colors = ['#F44336' if n in treated_nodes else '#2196F3' for n in G.nodes()]
nx.draw(G, pos, ax=axes[0], node_color=node_colors, node_size=120,
        edge_color='#BDBDBD', with_labels=False, arrows=False)
axes[0].set_title('Network Spillovers — SUTVA Violation
Red=treated, Blue=control
'
                  '(Control units near treated may be "protected")', fontsize=10)

# Timeline showing no-interference assumption
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 4); ax.axis('off')
ax.text(5, 3.5, 'SUTVA Components', ha='center', fontsize=12, fontweight='bold')
ax.text(0.5, 2.8, '1. No interference (stable across others's assignments)', fontsize=10)
ax.text(0.5, 2.3, '   OK: drug trials (patients don't share meds)', fontsize=9, color='green')
ax.text(0.5, 1.9, '   Violated: vaccines (herd immunity), job training (market wages)', fontsize=9, color='red')
ax.text(0.5, 1.3, '2. No hidden versions of treatment', fontsize=10)
ax.text(0.5, 0.9, '   OK: binary dose (treated vs not)', fontsize=9, color='green')
ax.text(0.5, 0.5, '   Violated: "job training" (many curricula, durations)', fontsize=9, color='red')

plt.tight_layout()
plt.savefig('01_sutva.png', dpi=110, bbox_inches='tight')
plt.show()

## 3. Estimands: ATE, ATT, ATC

Which population do you care about?

| Estimand | Formula | When to use |
|----------|---------|-------------|
| ATE | E[Y(1) − Y(0)] | Policy applied to whole population |
| ATT | E[Y(1) − Y(0) \| T=1] | Evaluating an existing program on participants |
| ATC | E[Y(1) − Y(0) \| T=0] | Would untreated benefit if we expanded? |

Different estimands require different identification assumptions and estimators. Propensity score matching targets ATT; IPW-ATE targets the whole distribution.

In [ ]:
# Visualize ATE vs ATT with heterogeneous effects
fig, ax = plt.subplots(figsize=(9, 4))

np.random.seed(3)
n_pop = 2000
# Propensity to be treated correlates with benefit (cream-skimming)
latent = np.random.normal(0, 1, n_pop)
p_select = 1 / (1 + np.exp(-latent))
treated_pop = np.random.binomial(1, p_select).astype(bool)
# True TE is higher for those selected into treatment
te_pop = 1 + 1.5 * latent + np.random.normal(0, 0.3, n_pop)

ax.hist(te_pop[~treated_pop], bins=50, density=True, alpha=0.6,
        color='#2196F3', label=f'Control (ATC = {te_pop[~treated_pop].mean():.2f})')
ax.hist(te_pop[treated_pop],  bins=50, density=True, alpha=0.6,
        color='#F44336', label=f'Treated (ATT = {te_pop[treated_pop].mean():.2f})')
ax.axvline(te_pop.mean(), color='black', lw=2, label=f'ATE = {te_pop.mean():.2f}')
ax.axvline(0, color='gray', ls=':', lw=1)
ax.set_xlabel('Individual Treatment Effect τ_i')
ax.set_ylabel('Density')
ax.set_title('Heterogeneous Effects: ATE ≠ ATT ≠ ATC')
ax.legend()
plt.tight_layout()
plt.savefig('01_estimands.png', dpi=110, bbox_inches='tight')
plt.show()
print(f"ATE={te_pop.mean():.3f}  ATT={te_pop[treated_pop].mean():.3f}  ATC={te_pop[~treated_pop].mean():.3f}")

## 4. DAG Notation and Path Types

A Directed Acyclic Graph (DAG) encodes causal assumptions as directed edges (X → Y = "X causally affects Y"). Three fundamental path structures determine which variables we need to control for:

| Pattern | Structure | Blocks? | d-separation rule |
|---------|-----------|---------|------------------|
| Chain | X → M → Y | Controlling M blocks X→Y | Block M |
| Fork (confounder) | X ← C → Y | C creates spurious X-Y correlation | Block C |
| Collider | X → C ← Y | C is inactive; conditioning *opens* the path | Don't condition on C |

The **backdoor criterion** (Pearl): a set Z blocks all backdoor paths from T to Y without blocking any directed paths, and Z contains no descendants of T. Then conditioning on Z identifies ATE.

In [ ]:
def draw_dag(edges, highlights=None, title='', figsize=(7, 4)):
    """Draw a causal DAG with optional highlighted paths."""
    G = nx.DiGraph()
    G.add_edges_from(edges)

    fig, ax = plt.subplots(figsize=figsize)
    pos = nx.spring_layout(G, seed=42, k=2)

    highlight_edges = highlights or []
    normal_edges = [e for e in G.edges() if e not in highlight_edges]

    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=1800,
                           node_color='#E3F2FD', edgecolors='#1565C0', linewidths=2)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=11, font_weight='bold')
    nx.draw_networkx_edges(G, pos, edgelist=normal_edges, ax=ax,
                           arrows=True, arrowsize=25, width=2,
                           edge_color='#555', connectionstyle='arc3,rad=0.1')
    if highlight_edges:
        nx.draw_networkx_edges(G, pos, edgelist=highlight_edges, ax=ax,
                               arrows=True, arrowsize=25, width=3,
                               edge_color='#F44336', connectionstyle='arc3,rad=0.1')
    ax.set_title(title, fontsize=11)
    ax.axis('off')
    return fig


# Classic confounded observational study DAG
# U = unobserved severity; Age → Treatment & Outcome; Treatment → Outcome
edges_obs = [
    ('Age', 'Treatment'),
    ('Age', 'Outcome'),
    ('Severity', 'Treatment'),
    ('Severity', 'Outcome'),
    ('Treatment', 'Outcome'),
]

fig = draw_dag(
    edges_obs,
    highlights=[('Age', 'Outcome'), ('Severity', 'Treatment')],
    title='Observational DAG: Confounders Age & Severity
(red = backdoor paths through confounders)',
    figsize=(8, 4)
)
plt.tight_layout()
plt.savefig('01_dag_confounders.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# Collider bias demonstration
# Conditioning on a collider opens a spurious path
np.random.seed(5)
n = 5000

# True DGP: talent ⊥ beauty (independent)
talent = np.random.normal(0, 1, n)
beauty = np.random.normal(0, 1, n)

# Collider: selection = f(talent, beauty) — only high scorers in dataset
selected = (talent + beauty > 1.0)

# Unconditional correlation
r_all = np.corrcoef(talent, beauty)[0,1]
# Conditional correlation (within selected set)
r_sel = np.corrcoef(talent[selected], beauty[selected])[0,1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (mask, label, r) in zip(axes, [
    (np.ones(n, bool), f'Full population (r={r_all:.3f})', r_all),
    (selected, f'Selected population (r={r_sel:.3f})', r_sel),
]):
    ax.scatter(talent[mask], beauty[mask], alpha=0.15, s=10, color='#2196F3')
    ax.set_xlabel('Talent'); ax.set_ylabel('Beauty')
    ax.set_title(label + '
Conditioning on collider (selection) opens spurious path')

plt.tight_layout()
plt.savefig('01_collider_bias.png', dpi=110, bbox_inches='tight')
plt.show()
print(f"Unconditional r(talent, beauty) = {r_all:.3f}  (truly independent)")
print(f"Conditional r(talent, beauty | selected) = {r_sel:.3f}  ← collider bias!")

## 5. Building a DAG for a Real Study

Draw the DAG *before* data analysis. Force yourself to commit to the causal structure. Every arrow is a claim. Missing arrows are also claims (no direct effect).

In [ ]:
# Example: estimating the effect of a job training program
# on earnings, using an observational dataset

# Variables:
#   T = training (treatment)
#   Y = earnings 2 years later (outcome)
#   A = age (confounder — affects both enrollment and earnings)
#   E = education (confounder)
#   M = employed at baseline (mediator — post-treatment! don't condition on this)
#   U = motivation (unobserved confounder)

training_dag_edges = [
    ('Age', 'Training'),
    ('Age', 'Earnings'),
    ('Education', 'Training'),
    ('Education', 'Earnings'),
    ('Motivation', 'Training'),   # unobserved
    ('Motivation', 'Earnings'),   # unobserved
    ('Training', 'Employed_t1'),  # mediator — don't condition!
    ('Employed_t1', 'Earnings'),
    ('Training', 'Earnings'),     # direct effect
]

fig = draw_dag(
    training_dag_edges,
    title='Job Training DAG
Adjustment set: {Age, Education} — NOT Employed_t1 (mediator)',
    figsize=(9, 5)
)
plt.tight_layout()
plt.savefig('01_training_dag.png', dpi=110, bbox_inches='tight')
plt.show()

print("Backdoor adjustment set: {Age, Education}")
print("Motivation is unobserved — if unmeasured confounding is large,")
print("  no adjustment set fully closes the backdoor.")
print("Employed_t1 is a MEDIATOR (post-treatment) — conditioning on it")
print("  would block part of the treatment effect and bias the estimate.")